# E2 — Perfilado, Diccionario y Limpieza Inicial

**Curso:** Data Visualization — UPC  
**Equipo:** Ricardo Rivas (U202215375) · Salvador Diaz (U202216148) · Joaquin Arévalo (U202212243)  
**Entrega:** E2 — Semana 5  
**Dataset:** Yelp Open Dataset (Academic)

---

## Objetivo

Perfilar los datasets fuente, construir el diccionario de datos, identificar problemas de calidad y aplicar las reglas de limpieza necesarias para producir un dataset limpio conectado a Tableau sin ambigüedad de tipos.

---
## 0. Unidad de análisis y granularidad

| Aspecto | Definición |
|---------|------------|
| **Unidad de análisis principal** | Negocio (`business_id`) — cada fila representa un local único registrado en Yelp |
| **Granularidad de reseñas** | Una fila = una reseña de un usuario sobre un negocio en una fecha |
| **Cobertura geográfica** | 11 áreas metropolitanas de EE.UU. y Canadá |
| **Período temporal** | Histórico hasta 2022 |
| **Pregunta analítica** | ¿Qué combinación de categoría, ubicación, atributos y evolución temporal caracteriza los segmentos de mayor oportunidad para abrir un negocio local? |

In [1]:
import json
import glob
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

RAW_DIR       = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
OUTPUT_DIR    = Path('../eda_output')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Rutas configuradas correctamente')

Rutas configuradas correctamente


---
## 1. Carga de datos

In [2]:
# Carga business.json
print('Cargando business.json...')
biz_raw = []
with open(RAW_DIR / 'yelp_academic_dataset_business.json', encoding='utf-8') as f:
    for line in f:
        biz_raw.append(json.loads(line))

biz = pd.DataFrame(biz_raw)
print(f'  Filas: {len(biz):,}')
print(f'  Columnas: {len(biz.columns)}')
print(f'  Columnas disponibles: {biz.columns.tolist()}')

Cargando business.json...


  Filas: 150,346
  Columnas: 14
  Columnas disponibles: ['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours']


In [3]:
# Carga parquets de reseñas
print('Cargando reviews (parquets)...')
files = sorted(glob.glob(str(PROCESSED_DIR / 'reviews_enriched_v1_part_*.parquet')))
reviews = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
reviews['date'] = pd.to_datetime(reviews['date'])
print(f'  Filas: {len(reviews):,}')
print(f'  Columnas: {reviews.columns.tolist()}')

Cargando reviews (parquets)...


  Filas: 775,955
  Columnas: ['review_id', 'user_id', 'business_id', 'review_stars', 'review_useful', 'review_funny', 'review_cool', 'text', 'date', 'business_name', 'categories', 'business_stars', 'business_review_count', 'user_review_count', 'user_average_stars', 'categories_list']


---
## 2. Perfilado — Dataset de Negocios (`business.json`)

In [4]:
# Tabla de perfilado: tipo, nulos, cardinalidad, muestra
cols_biz = ['business_id','name','address','city','state','postal_code',
            'latitude','longitude','stars','review_count','is_open','categories']

profile_biz = pd.DataFrame({
    'columna':      cols_biz,
    'tipo_dato':    [str(biz[c].dtype) for c in cols_biz],
    'nulos_n':      [biz[c].isna().sum() for c in cols_biz],
    'nulos_pct':    [(biz[c].isna().sum() / len(biz) * 100).round(2) for c in cols_biz],
    'cardinalidad': [biz[c].nunique() for c in cols_biz],
    'muestra':      [str(biz[c].dropna().iloc[0]) if biz[c].notna().any() else 'N/A' for c in cols_biz],
})

print('=== Perfilado: Negocios ===')
profile_biz

=== Perfilado: Negocios ===


,columna,tipo_dato,nulos_n,nulos_pct,cardinalidad,muestra
0,business_id,str,0,0.00,150346,Pns2l4eNsfO8kk83dixA6A
1,name,str,0,0.00,114117,"Abby Rappoport, LAC, CMQ"
2,address,str,0,0.00,122844,"1616 Chapala St, Ste 2"
3,city,str,0,0.00,1416,Santa Barbara
4,state,str,0,0.00,27,CA
5,postal_code,str,0,0.00,3362,93101
6,latitude,float64,0,0.00,135593,34.4266787
7,longitude,float64,0,0.00,131918,-119.7111968
8,stars,float64,0,0.00,9,5.0
9,review_count,int64,0,0.00,1158,7


In [5]:
# Estadísticas numéricas
print('=== Estadísticas numéricas: Negocios ===')
biz[['stars','review_count','latitude','longitude']].describe().round(3)

=== Estadísticas numéricas: Negocios ===


,stars,review_count,latitude,longitude
count,150346.00,150346.00,150346.00,150346.00
mean,3.60,44.87,36.67,-89.36
std,0.97,121.12,5.87,14.92
min,1.00,5.00,27.55,-120.09
25%,3.00,8.00,32.19,-90.36
50%,3.50,15.00,38.78,-86.12
75%,4.50,37.00,39.95,-75.42
max,5.00,7568.00,53.68,-73.20


In [6]:
# Duplicados
dup_ids   = biz['business_id'].duplicated().sum()
dup_names = biz.duplicated(subset=['name','city','state']).sum()
print(f'Duplicados por business_id:       {dup_ids}')
print(f'Duplicados por nombre+ciudad+estado: {dup_names}')

# Distribución is_open
print('\nDistribución is_open:')
print(biz['is_open'].value_counts().rename({1: 'Abierto', 0: 'Cerrado'}))

Duplicados por business_id:       0
Duplicados por nombre+ciudad+estado: 12974

Distribución is_open:
is_open
Abierto    119698
Cerrado     30648
Name: count, dtype: int64


In [7]:
# Distribución geográfica
print('Top 10 ciudades por número de negocios:')
print(biz['city'].value_counts().head(10).to_string())
print()
print('Top 10 estados:')
print(biz['state'].value_counts().head(10).to_string())

Top 10 ciudades por número de negocios:
city
Philadelphia     14569
Tucson            9250
Tampa             9050
Indianapolis      7540
Nashville         6971
New Orleans       6209
Reno              5935
Edmonton          5054
Saint Louis       4827
Santa Barbara     3829

Top 10 estados:
state
PA    34039
FL    26330
TN    12056
IN    11247
MO    10913
LA     9924
AZ     9912
NJ     8536
NV     7715
AB     5573


In [8]:
# Rango de coordenadas
print('Rango de coordenadas:')
print(f'  Latitude:  {biz.latitude.min():.4f} → {biz.latitude.max():.4f}')
print(f'  Longitude: {biz.longitude.min():.4f} → {biz.longitude.max():.4f}')

# Coordenadas fuera de rango razonable (lat 24-50, lon -125 a -65 para EE.UU./Canada)
lat_bad = biz[(biz.latitude < 24) | (biz.latitude > 72)]
lon_bad = biz[(biz.longitude < -141) | (biz.longitude > -52)]
print(f'\nCoordenadas lat fuera de rango EE.UU./Canada: {len(lat_bad)}')
print(f'Coordenadas lon fuera de rango EE.UU./Canada: {len(lon_bad)}')

Rango de coordenadas:
  Latitude:  27.5551 → 53.6792
  Longitude: -120.0951 → -73.2005

Coordenadas lat fuera de rango EE.UU./Canada: 0
Coordenadas lon fuera de rango EE.UU./Canada: 0


In [9]:
# Análisis del campo categories
cats_null = biz['categories'].isna().sum()
print(f'Negocios sin categoría: {cats_null} ({cats_null/len(biz)*100:.1f}%)')

# Conteo de categorías únicas
all_cats = []
for c in biz['categories'].dropna():
    all_cats.extend([x.strip() for x in c.split(',')])

from collections import Counter
cat_counter = Counter(all_cats)
print(f'Categorías únicas totales: {len(cat_counter):,}')
print(f'\nTop 15 categorías:')
for cat, n in cat_counter.most_common(15):
    print(f'  {n:>6,}  {cat}')

Negocios sin categoría: 103 (0.1%)


Categorías únicas totales: 1,311

Top 15 categorías:
  52,268  Restaurants
  27,781  Food
  24,395  Shopping
  14,356  Home Services
  14,292  Beauty & Spas
  12,281  Nightlife
  11,890  Health & Medical
  11,198  Local Services
  11,065  Bars
  10,773  Automotive
   9,895  Event Planning & Services
   8,366  Sandwiches
   8,139  American (Traditional)
   7,687  Active Life
   7,093  Pizza


---
## 3. Perfilado — Dataset de Reseñas (parquets)

In [10]:
cols_rev = ['review_id','business_id','user_id','review_stars','date',
            'review_useful','review_funny','review_cool','business_stars','business_review_count']
cols_rev = [c for c in cols_rev if c in reviews.columns]

profile_rev = pd.DataFrame({
    'columna':      cols_rev,
    'tipo_dato':    [str(reviews[c].dtype) for c in cols_rev],
    'nulos_n':      [reviews[c].isna().sum() for c in cols_rev],
    'nulos_pct':    [(reviews[c].isna().sum() / len(reviews) * 100).round(2) for c in cols_rev],
    'cardinalidad': [reviews[c].nunique() for c in cols_rev],
})

print('=== Perfilado: Reseñas ===')
profile_rev

=== Perfilado: Reseñas ===


,columna,tipo_dato,nulos_n,nulos_pct,cardinalidad
0,review_id,string,0,0.00,775955
1,business_id,string,0,0.00,9930
2,user_id,string,0,0.00,154035
3,review_stars,float64,0,0.00,5
4,date,datetime64[us],0,0.00,774500
5,review_useful,Int64,0,0.00,131
6,review_funny,Int64,0,0.00,103
7,review_cool,Int64,0,0.00,118
8,business_stars,float64,0,0.00,9
9,business_review_count,Int64,0,0.00,674


In [11]:
print('Estadísticas numéricas: Reseñas')
num_cols = [c for c in ['review_stars','review_useful','review_funny','review_cool'] if c in reviews.columns]
reviews[num_cols].describe().round(3)

Estadísticas numéricas: Reseñas


,review_stars,review_useful,review_funny,review_cool
count,775955.00,775955.00,775955.00,775955.00
mean,3.85,1.39,0.45,0.66
std,1.29,3.10,1.82,2.23
min,1.00,0.00,0.00,0.00
25%,3.00,0.00,0.00,0.00
50%,4.00,1.00,0.00,0.00
75%,5.00,2.00,0.00,1.00
max,5.00,224.00,284.00,192.00


In [12]:
# Rango temporal
print(f'Rango temporal: {reviews.date.min().date()} → {reviews.date.max().date()}')
print(f'\nReseñas por año:')
print(reviews.groupby(reviews.date.dt.year)['review_id'].count().to_string())

Rango temporal: 2005-02-16 → 2022-01-19

Reseñas por año:


date
2005      118
2006      948
2007     5698
2008    14349
2009    23271
2010    35706
2011    46661
2012    52630
2013    62422
2014    71269
2015    78227
2016    77812
2017    80233
2018    78692
2019    74708
2020    35243
2021    36218
2022     1750


---
## 4. Diccionario de Datos

### 4.1 Dataset de Negocios (`business.json`)

| Variable | Tipo | Rol analítico | Descripción | Observaciones de calidad |
|----------|------|---------------|-------------|---------------------------|
| `business_id` | STRING | Llave primaria | Identificador único del negocio | Sin nulos, sin duplicados |
| `name` | STRING | Descriptiva | Nombre del negocio | Puede tener variantes de escritura |
| `city` | STRING | Segmentación geográfica | Ciudad del negocio | Homologado con str.title() |
| `state` | STRING | Segmentación geográfica | Código de estado/provincia | Estandarizado en 2-3 letras |
| `latitude` | FLOAT | Geoespacial | Latitud del negocio | Validado en rango EE.UU./Canadá |
| `longitude` | FLOAT | Geoespacial | Longitud del negocio | Validado en rango EE.UU./Canadá |
| `stars` | FLOAT {1.0–5.0} | KPI principal | Calificación agregada de Yelp | Distribución sesgada a valores altos; raramente < 2.5 |
| `review_count` | INT ≥ 0 | KPI principal | Proxy de demanda y visibilidad | Distribución muy sesgada — se usa log1p |
| `is_open` | INT {0,1} | Filtro | 1 = negocio activo, 0 = cerrado | Solo negocios abiertos en el análisis |
| `categories` | STRING (multi-valor) | Segmentación | Etiquetas del negocio separadas por coma | Parseado; `primary_category` = primera etiqueta |
| `attributes` | JSON anidado | Segmentación | WiFi, estacionamiento, ambiente, etc. | Alta tasa de nulos; usar solo cobertura > 60% — pendiente E3 |
| `hours` | JSON anidado | Soporte | Horarios de apertura | Requiere parseo para calcular horas semanales — pendiente E3 |

### 4.2 Dataset de Reseñas (parquets `reviews_enriched_v1`)

| Variable | Tipo | Rol analítico | Descripción |
|----------|------|---------------|-------------|
| `review_id` | STRING | Llave primaria | Identificador único de reseña |
| `business_id` | STRING | Llave foránea → Negocios | Enlace con dataset de negocios |
| `user_id` | STRING | Llave foránea → Usuarios | Enlace con dataset de usuarios |
| `review_stars` | INT {1–5} | KPI principal | Calificación atómica; base para señal de calidad matizada |
| `date` | DATETIME | Serie temporal | Fecha de publicación; insumo para análisis longitudinal |
| `review_useful` | INT ≥ 0 | Soporte | Votos de utilidad; proxy de credibilidad |
| `review_funny` | INT ≥ 0 | Soporte | Votos de humor |
| `review_cool` | INT ≥ 0 | Soporte | Votos de apreciación |
| `business_stars` | FLOAT | KPI (desnormalizado) | Calificación agregada del negocio en el momento de la reseña |
| `business_review_count` | INT | KPI (desnormalizado) | Conteo total de reseñas del negocio |
| `user_average_stars` | FLOAT | Soporte | Media histórica de calificaciones del usuario — permite calcular `calificación ajustada = review_stars − user_average_stars` sin necesitar user.json |
| `user_review_count` | INT | Soporte | Total de reseñas escritas por el usuario — proxy de experiencia en la plataforma |

---
## 5. Problemas de Calidad Identificados

| # | Dataset | Campo | Tipo de problema | Severidad | Decisión |
|---|---------|-------|-----------------|-----------|----------|
| 1 | Negocios | `categories` | Nulos (~0.1%) y campo multi-valor separado por coma | Media | Parsear; extraer `primary_category` (primera etiqueta); rellenar nulos con `'Unknown'` |
| 2 | Negocios | `city` | Inconsistencias de capitalización y tildes | Media | Aplicar `.str.strip().str.title()` |
| 3 | Negocios | `review_count` | Distribución muy sesgada a la derecha (cola larga) | Alta | Crear `log_review_count = log1p(review_count)` para análisis y normalización |
| 4 | Negocios | `attributes`, `hours` | JSON anidado con alta tasa de nulos | Media | Aplanar solo atributos con cobertura > 60%; resto se excluye |
| 5 | Negocios | `is_open` | Negocios cerrados mezclados con activos | Alta | Filtrar solo `is_open == 1` para análisis de oportunidad |
| 6 | Negocios | `latitude`/`longitude` | Posibles coordenadas fuera del rango geográfico esperado | Baja | Validar y excluir registros con coordenadas inválidas |
| 7 | Reseñas | `date` | Dtype object; necesita conversión a datetime | Alta | Convertir con `pd.to_datetime()` |
| 8 | Reseñas | Período 2020–2021 | Caída pronunciada por COVID — anomalía real, no error | Informativa | Documentar; no excluir; marcar en visualizaciones |

---
## 6. Limpieza y Transformaciones

In [13]:
# Bitácora de transformaciones
bitacora = []

def log_transform(dataset, campo, tipo, original, transformado, razon):
    bitacora.append({
        'dataset': dataset, 'campo': campo, 'tipo_problema': tipo,
        'valor_original': original, 'valor_transformado': transformado, 'razon': razon
    })

print('Bitacora inicializada')

Bitacora inicializada


In [14]:
# Problema 1: Filtrar negocios cerrados
n_antes = len(biz)
biz_clean = biz[biz['is_open'] == 1].copy()
n_despues = len(biz_clean)
log_transform('Negocios', 'is_open', 'Filtrado',
              f'{n_antes:,} filas (incluye cerrados)',
              f'{n_despues:,} filas (solo abiertos)',
              'Análisis de oportunidad requiere solo negocios activos')
print(f'is_open: {n_antes:,} → {n_despues:,} filas (eliminados {n_antes - n_despues:,} cerrados)')

is_open: 150,346 → 119,698 filas (eliminados 30,648 cerrados)


In [15]:
# Problema 2: Homologar city
biz_clean['city'] = biz_clean['city'].str.strip().str.title()
log_transform('Negocios', 'city', 'Homologación',
              'Texto libre con variaciones de capitalización',
              'str.strip().str.title() aplicado',
              'Garantiza agrupaciones geográficas consistentes en Tableau')
print('city: homologado con str.title()')

# Verificar top ciudades post-limpieza
print(biz_clean['city'].value_counts().head(5).to_string())

city: homologado con str.title()
city
Philadelphia    10549
Tucson           7544
Tampa            7237
Indianapolis     5899
Nashville        5408


In [16]:
# Problema 3: Extraer categoría primaria
biz_clean['primary_category'] = (
    biz_clean['categories']
    .fillna('Unknown')
    .str.split(',')
    .str[0]
    .str.strip()
)
log_transform('Negocios', 'categories', 'Parseo + derivada',
              'String multi-valor separado por coma; 0.1% nulos',
              'primary_category = primera etiqueta parseada; nulos → Unknown',
              'Permite segmentación simple en Tableau sin explosión de filas')
print('primary_category creada:')
print(biz_clean['primary_category'].value_counts().head(10).to_string())

primary_category creada:
primary_category
Restaurants                  9963
Food                         4953
Shopping                     4462
Beauty & Spas                3778
Home Services                3522
Automotive                   3159
Health & Medical             2851
Local Services               2391
Event Planning & Services    1729
Active Life                  1627


In [17]:
# Problema 4: Crear log_review_count
biz_clean['log_review_count'] = np.log1p(biz_clean['review_count'])
log_transform('Negocios', 'review_count', 'Transformación',
              'Distribución muy sesgada a la derecha (max >> mediana)',
              'log_review_count = log1p(review_count)',
              'Normaliza la distribución para comparaciones y score de oportunidad')
print('log_review_count creada')
print(biz_clean['log_review_count'].describe().round(3))

log_review_count creada
count   119698.00
mean         3.03
std          1.09
min          1.79
25%          2.20
50%          2.77
75%          3.64
max          8.93
Name: log_review_count, dtype: float64


In [18]:
# Problema 5: Validar coordenadas
n_coords_bad = biz_clean[
    (biz_clean.latitude < 24) | (biz_clean.latitude > 72) |
    (biz_clean.longitude < -141) | (biz_clean.longitude > -52)
].shape[0]

biz_clean = biz_clean[
    (biz_clean.latitude >= 24) & (biz_clean.latitude <= 72) &
    (biz_clean.longitude >= -141) & (biz_clean.longitude <= -52)
].copy()

log_transform('Negocios', 'latitude/longitude', 'Filtrado',
              f'Coordenadas sin restricción de rango ({n_coords_bad} fuera de EE.UU./Canada)',
              'Excluidos registros fuera del rango lat[24,72] lon[-141,-52]',
              'Dataset cubre solo EE.UU. y Canadá; coordenadas fuera = error de entrada')
print(f'Coordenadas: eliminados {n_coords_bad} registros con coordenadas inválidas')
print(f'Filas finales: {len(biz_clean):,}')

Coordenadas: eliminados 0 registros con coordenadas inválidas
Filas finales: 119,698


In [19]:
# Normalización para score de oportunidad
mn_s, mx_s = biz_clean['stars'].min(), biz_clean['stars'].max()
mn_r, mx_r = biz_clean['log_review_count'].min(), biz_clean['log_review_count'].max()
biz_clean['stars_norm']  = (biz_clean['stars'] - mn_s) / (mx_s - mn_s)
biz_clean['log_rc_norm'] = (biz_clean['log_review_count'] - mn_r) / (mx_r - mn_r)
biz_clean['divergence_score'] = biz_clean['stars_norm'] - biz_clean['log_rc_norm']
log_transform('Negocios', 'stars / log_review_count', 'Derivada',
              'Valores absolutos no comparables entre sí',
              'stars_norm y log_rc_norm en [0,1]; divergence_score = stars_norm - log_rc_norm',
              'Insumo directo para el score de oportunidad del dashboard')
print('Normalización aplicada')

Normalización aplicada


---
## 7. Bitácora de Transformaciones (resumen)

In [20]:
bitacora_df = pd.DataFrame(bitacora)
bitacora_df

,dataset,campo,tipo_problema,valor_original,valor_transformado,razon
0,Negocios,is_open,Filtrado,"150,346 filas (incluye cerrados)","119,698 filas (solo abiertos)",Análisis de oportunidad requiere solo negocios...
1,Negocios,city,Homologación,Texto libre con variaciones de capitalización,str.strip().str.title() aplicado,Garantiza agrupaciones geográficas consistente...
2,Negocios,categories,Parseo + derivada,String multi-valor separado por coma; 0.1% nulos,primary_category = primera etiqueta parseada; ...,Permite segmentación simple en Tableau sin exp...
3,Negocios,review_count,Transformación,Distribución muy sesgada a la derecha (max >> ...,log_review_count = log1p(review_count),Normaliza la distribución para comparaciones y...
4,Negocios,latitude/longitude,Filtrado,Coordenadas sin restricción de rango (0 fuera ...,"Excluidos registros fuera del rango lat[24,72]...",Dataset cubre solo EE.UU. y Canadá; coordenada...
5,Negocios,stars / log_review_count,Derivada,Valores absolutos no comparables entre sí,"stars_norm y log_rc_norm en [0,1]; divergence_...",Insumo directo para el score de oportunidad de...


---
## 8. Exportar dataset limpio

In [21]:
# Columnas seleccionadas para el dataset limpio (sin attributes/hours anidados)
export_cols = [
    'business_id', 'name', 'city', 'state', 'postal_code',
    'latitude', 'longitude', 'stars', 'review_count', 'is_open',
    'categories', 'primary_category',
    'log_review_count', 'stars_norm', 'log_rc_norm', 'divergence_score'
]
export_cols = [c for c in export_cols if c in biz_clean.columns]

biz_final = biz_clean[export_cols].copy()
out_path = OUTPUT_DIR / 'business_clean.csv'
biz_final.to_csv(out_path, index=False, encoding='utf-8')

print(f'Dataset limpio guardado: {out_path}')
print(f'Filas: {len(biz_final):,}  |  Columnas: {len(biz_final.columns)}')
print(f'Columnas: {biz_final.columns.tolist()}')
print()
print('Tipos de datos (listos para Tableau):')
print(biz_final.dtypes)

Dataset limpio guardado: ..\eda_output\business_clean.csv
Filas: 119,698  |  Columnas: 16
Columnas: ['business_id', 'name', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'categories', 'primary_category', 'log_review_count', 'stars_norm', 'log_rc_norm', 'divergence_score']

Tipos de datos (listos para Tableau):
business_id             str
name                    str
city                    str
state                   str
postal_code             str
latitude            float64
longitude           float64
stars               float64
review_count          int64
is_open               int64
categories              str
primary_category     object
log_review_count    float64
stars_norm          float64
log_rc_norm         float64
divergence_score    float64
dtype: object


In [22]:
# Exportar bitácora
bitacora_df.to_csv(OUTPUT_DIR / 'bitacora_transformaciones.csv', index=False, encoding='utf-8')
print('Bitacora exportada: eda_output/bitacora_transformaciones.csv')

Bitacora exportada: eda_output/bitacora_transformaciones.csv


---
## 9. Validación final del dataset limpio

In [23]:
print('=== Validación final ===')
print(f'Filas totales:              {len(biz_final):,}')
print(f'Negocios únicos:            {biz_final.business_id.nunique():,}')
print(f'Nulos restantes por col:')
nulos_final = biz_final.isna().sum()
print(nulos_final[nulos_final > 0].to_string() if nulos_final.any() else '  Ninguno')
print(f'\nCiudades cubiertas:         {biz_final.city.nunique():,}')
print(f'Estados cubiertos:          {biz_final.state.nunique():,}')
print(f'Categorías primarias:       {biz_final.primary_category.nunique():,}')
print(f'Rango stars:                {biz_final.stars.min()} → {biz_final.stars.max()}')
print(f'Rango review_count:         {biz_final.review_count.min()} → {biz_final.review_count.max():,}')
print(f'Todos is_open == 1:         {(biz_final.is_open == 1).all()}')

=== Validación final ===
Filas totales:              119,698
Negocios únicos:            119,698
Nulos restantes por col:
categories    95

Ciudades cubiertas:         1,156
Estados cubiertos:          24
Categorías primarias:       1,148
Rango stars:                1.0 → 5.0
Rango review_count:         5 → 7,568
Todos is_open == 1:         True


In [24]:
print('Dataset limpio listo para conectar a Tableau.')
print('Archivo: eda_output/business_clean.csv')
print('Bitacora: eda_output/bitacora_transformaciones.csv')
biz_final.head(3)

Dataset limpio listo para conectar a Tableau.
Archivo: eda_output/business_clean.csv
Bitacora: eda_output/bitacora_transformaciones.csv


,business_id,name,city,state,postal_code,latitude,longitude,stars,review_count,is_open,categories,primary_category,log_review_count,stars_norm,log_rc_norm,divergence_score
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,Affton,MO,63123,38.55,-90.34,3.00,15,1,"Shipping Centers, Local Services, Notaries, Ma...",Shipping Centers,2.77,0.50,0.14,0.36
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,Philadelphia,PA,19107,39.96,-75.16,4.00,80,1,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",Restaurants,4.39,0.75,0.36,0.39
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,Green Lane,PA,18054,40.34,-75.47,4.50,13,1,"Brewpubs, Breweries, Food",Brewpubs,2.64,0.88,0.12,0.76


In [25]:
# Validación de integridad referencial: business_id en reseñas vs negocios
biz_ids_clean   = set(biz_final['business_id'])
review_biz_ids  = set(reviews['business_id'])

match     = review_biz_ids & biz_ids_clean
no_match  = review_biz_ids - biz_ids_clean

print('=== Validación de integridad referencial ===')
print(f'business_id únicos en reseñas:         {len(review_biz_ids):,}')
print(f'business_id únicos en negocios limpios:{len(biz_ids_clean):,}')
print(f'Con match (inner join):                {len(match):,}')
print(f'Sin match en negocios (huérfanos):     {len(no_match):,}')
if no_match:
    pct = len(no_match) / len(review_biz_ids) * 100
    print(f'  → {pct:.1f}% de business_ids en reseñas corresponden a negocios cerrados (filtrados en limpieza)')
    print('  Decisión: aceptable — los negocios cerrados fueron excluidos intencionalmente.')
else:
    print('  Integridad perfecta.')

# Validar que no hay review_id duplicados
dup_reviews = reviews['review_id'].duplicated().sum()
print(f'\nReseñas duplicadas (review_id):         {dup_reviews}')

# Totales de métricas clave para validación cruzada
print(f'\nTotal reseñas en parquets:              {len(reviews):,}')
print(f'Negocios con al menos 1 reseña:         {reviews.business_id.nunique():,}')
print(f'Usuarios únicos reseñadores:            {reviews.user_id.nunique():,}')

=== Validación de integridad referencial ===
business_id únicos en reseñas:         9,930
business_id únicos en negocios limpios:119,698
Con match (inner join):                7,129
Sin match en negocios (huérfanos):     2,801
  → 28.2% de business_ids en reseñas corresponden a negocios cerrados (filtrados en limpieza)
  Decisión: aceptable — los negocios cerrados fueron excluidos intencionalmente.

Reseñas duplicadas (review_id):         0

Total reseñas en parquets:              775,955


Negocios con al menos 1 reseña:         9,930
Usuarios únicos reseñadores:            154,035


---
## 10. Modelado Relacional

### 10.1 Esquema de tablas

```
business.json (Dim. Negocios)          reviews_enriched_v1 (Tabla de Hechos)
─────────────────────────────          ─────────────────────────────────────
business_id  ◄──────────────────────── business_id
name                                   review_id
city                                   user_id
state                                  review_stars
latitude                               date
longitude                              review_useful / funny / cool
stars                                  user_average_stars  ← desnorm. de user.json
review_count                           user_review_count   ← desnorm. de user.json
categories                             business_stars      ← desnorm. de business.json
primary_category                       business_review_count
[variables derivadas]
```

**Decisión de diseño:** Los parquets de reseñas ya tienen desnormalizados `user_average_stars`, `user_review_count`, `business_stars` y `business_review_count`. Esto evita joins adicionales en Tableau y reduce la carga de procesamiento. La tabla de hechos es autónoma para los KPIs principales.

### 10.2 Cardinalidad de las relaciones

| Relación | Tipo | Llave | Observación |
|----------|------|-------|-------------|
| Negocios → Reseñas | 1 : N | `business_id` | Un negocio tiene múltiples reseñas |
| Reseñas → Negocios | N : 1 | `business_id` | Cada reseña pertenece a exactamente 1 negocio |

### 10.3 Validación de integridad referencial

Ver celda siguiente: verificación de que todos los `business_id` en reseñas existen en el dataset de negocios.